This code was run on google colabs A100 GPU with 80 GB RAM

In [ ]:
!pip install datasets

In [ ]:
import torch
from torch import nn
import torch.optim as optim
import torch.nn.functional as F
from transformers import T5ForConditionalGeneration, T5Tokenizer
import numpy as np
import datasets
from datasets import DatasetDict
from sklearn.model_selection import train_test_split
import random

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

import matplotlib.pyplot as plt


Load model

In [ ]:
model_name = 'jbochi/madlad400-3b-mt'
model = T5ForConditionalGeneration.from_pretrained(model_name)
tokenizer = T5Tokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/11.8G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/830 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.43M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/4.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.6M [00:00<?, ?B/s]

Load datasets

In [ ]:
gatitos: DatasetDict = datasets.load_dataset("google/smol", "gatitos__yue_zh") # type: ignore
smolsent: DatasetDict = datasets.load_dataset("google/smol", "smolsent__en_yue") # type: ignore
smoldoc: DatasetDict = datasets.load_dataset("google/smol", "smoldoc__en_yue") # type: ignore

README.md: 0.00B [00:00, ?B/s]

yue_zh.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

en_yue.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

en_yue.jsonl: 0.00B [00:00, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


Define Hyperparameters

In [ ]:
epochs = 4
batch_size = 8
max_len = 128

Create dataloaders

In [ ]:
class SmolsentDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return self.dataset.__len__()

    def __getitem__(self, idx):
        return self.dataset[idx]["trg"], self.dataset[idx]["src"]

smolsent_train_validation = smolsent["train"].train_test_split(train_size=0.9, shuffle=True, seed=2)
smolsent_train_test = smolsent_train_validation["train"].train_test_split(train_size=8/9, shuffle=True, seed=2)

smolsent_train = smolsent_train_test["train"]
smolsent_validation = smolsent_train_validation["test"]
smolsent_test = smolsent_train_test["test"]

smolsent_validation_dataset = SmolsentDataset(smolsent_validation)
smolsent_test_dataset = SmolsentDataset(smolsent_test)
smolsent_train_dataset = SmolsentDataset(smolsent_train)

Prepare dataset

In [ ]:
from functools import reduce
from operator import add
from datasets import Dataset, DatasetDict, concatenate_datasets

canto_smoldoc = reduce(add, smoldoc["train"]["trgs"], [])
labels_smoldoc = reduce(add, smoldoc["train"]["srcs"], [])

canto_smoldoc_tokenized = tokenizer(canto_smoldoc, add_special_tokens=True)
labels_smoldoc_tokenized = tokenizer(labels_smoldoc, add_special_tokens=True)

# remove to avoid going over max length since this could mess up training
filtered_pairs_smoldoc = [
    (c, l, c_tok, l_tok)
    for c, l, c_tok, l_tok in zip(
        canto_smoldoc,
        labels_smoldoc,
        canto_smoldoc_tokenized['input_ids'],
        labels_smoldoc_tokenized['input_ids']
    )
    if len(c_tok) <= 120 and len(l_tok) <= 120 # 120 instead of 128 because of special tokens
]

canto_filtered_smoldoc = [pair[0] for pair in filtered_pairs_smoldoc]
labels_filtered_smoldoc = [pair[1] for pair in filtered_pairs_smoldoc]
canto_ids_filtered_smoldoc = [pair[2] for pair in filtered_pairs_smoldoc]
labels_ids_filtered_smoldoc = [pair[3] for pair in filtered_pairs_smoldoc]


filtered_smoldoc_dataset = Dataset.from_dict({
    "trg": canto_filtered_smoldoc,
    "src": labels_filtered_smoldoc
})

filtered_train_validation_smoldoc = filtered_smoldoc_dataset.train_test_split(train_size=0.9, shuffle=True, seed=2)
filtered_train_test_smoldoc = filtered_train_validation_smoldoc["train"].train_test_split(train_size=8/9, shuffle=True, seed=2)

filtered_train_smoldoc = filtered_train_test_smoldoc["train"]
filtered_validation_smoldoc = filtered_train_validation_smoldoc["test"]
filtered_test_smoldoc = filtered_train_test_smoldoc["test"]

# Concat smolsent and smoldoc data
smol_train = concatenate_datasets([smolsent_train, filtered_train_smoldoc])
smol_validation = concatenate_datasets([smolsent_validation, filtered_validation_smoldoc])
smol_test = concatenate_datasets([smolsent_test, filtered_test_smoldoc])

def preprocess(batch):
  trg = [f"<2en> {t}" for t in batch["trg"]]

  model_inputs = tokenizer(
      trg, padding="max_length", truncation=True, max_length=max_len
  )
  labels = tokenizer(
      batch["src"], padding="max_length", truncation=True, max_length=max_len
  )

  model_inputs["labels"] = labels["input_ids"]
  return model_inputs

smolsent_train_dataset = smol_train.map(preprocess, batched=True)
smolsent_test_dataset = smol_test.map(preprocess, batched=True)
smolsent_validation_dataset = smol_validation.map(preprocess, batched=True)

smolsent_train_dataloader = DataLoader(smolsent_train_dataset, batch_size=batch_size, shuffle=True) # type: ignore
smolsent_validation_dataloader = DataLoader(smolsent_validation_dataset, batch_size=batch_size, shuffle=True) # type: ignore
smolsent_test_dataloader = DataLoader(smolsent_test_dataset, batch_size=batch_size, shuffle=True) # type: ignore

print(f"Train size: {len(smol_train)}")
print(f"Validation size: {len(smol_validation)}")
print(f"Test size: {len(smol_test)}")

Map:   0%|          | 0/6265 [00:00<?, ? examples/s]

Map:   0%|          | 0/784 [00:00<?, ? examples/s]

Map:   0%|          | 0/784 [00:00<?, ? examples/s]

Train size: 6265
Validation size: 784
Test size: 784


Define training arguments and train model

In [ ]:
from transformers import TrainingArguments, Trainer
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=epochs,
    report_to='wandb',
    logging_steps=10,
    eval_steps=50,
    eval_strategy='steps',
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    weight_decay=0.01,
    save_strategy='epoch',
    warmup_steps=100,
    run_name='151225_4_epochs'
)


In [ ]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset = smolsent_train_dataset,
    eval_dataset = smolsent_validation_dataset,
    data_collator=data_collator,
    tokenizer=tokenizer
)

print('Start training')
trainer.train()
print('End training')

/tmp/ipython-input-278756648.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Start training


Step,Training Loss
500,0.039400
1000,0.024200
1500,0.027800


End training
